[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/02_softmax.ipynb)

# 🟢 简单：实现 Softmax

请从零开始实现 **Softmax** 函数。

$$\text{softmax}(x_i) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

### 函数签名
```python
def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    ...
```

### 规则
- **禁止**使用 `torch.softmax`、`F.softmax` 或 `torch.nn.Softmax`
- 必须**数值稳定**（提示：在做 exp 之前先减去最大值）

### 示例
```
输入:  tensor([1., 2., 3.])
输出: tensor([0.0900, 0.2447, 0.6652])  # 和约为 1.0
```

In [ ]:
# 在 Colab 中安装 torch-judge（在 JupyterLab/Docker 中无操作）
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch

/Users/xiaohui/Documents/project/TorchCode/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [10]:
# ✏️ 在基础实现(可能会数值溢出)

def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:
    exps = torch.exp(x) # 将张量所有的值取指数
    return exps / torch.sum(exps, dim=dim, keepdim=True) # keepdim, 保留维度，但不是重复 [1.0, 2.0, 3.0] -> [6.] 而不是 6.

- 降低指数防止数值溢出
1. 找到指定维度上的最大值
2. 减去最大值，进行平移（指数减法等于除法，每个元素都减去最大的值， ）
3. 计算总和
4. 归一化

$$P_i' = \frac{e^{x_i + c}}{\sum e^{x_j + c}} = \frac{e^{x_i} \cdot e^c}{\sum (e^{x_j} \cdot e^c)}$$
$$P_i' = \frac{e^{x_i} \cdot \cancel{e^c}}{\cancel{e^c} \cdot \sum e^{x_j}} = \frac{e^{x_i}}{\sum e^{x_j}} = P_i$$

这里 $c$ 是 负最大值

In [ ]:
# ✏️ 稳定实现

def my_softmax(x: torch.Tensor, dim: int = -1) -> torch.Tensor:

    max_val = torch.max(x, dim=dim, keepdim=True)[0]     # max 返回最大值和最大值位置索引
    x_exp = torch.exp(x - max_val)
    partition_sum = torch.sum(x_exp, dim=dim, keepdim=True)
    
    return x_exp / partition_sum

In [ ]:
# 🧪 调试
x = torch.tensor([1.0, 2.0, 3.0])
print("输出:  ", my_softmax(x, dim=-1))
print("总和:  ", my_softmax(x, dim=-1).sum())  # 应该接近 1.0
print("参考:  ", torch.softmax(x, dim=-1))

torch.return_types.max(
values=tensor([3.]),
indices=tensor([2]))
输出:   tensor([0.0900, 0.2447, 0.6652])
总和:   tensor(1.)
参考:   tensor([0.0900, 0.2447, 0.6652])


In [ ]:
# ✅ 提交验证
from torch_judge import check
check("softmax")